In [8]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import make_scorer, r2_score, mean_absolute_error
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance

plt.rcParams["figure.figsize"]=(8,5)
pd.set_option("display.max_colwidth", 140)


In [9]:
# Eğer DF'ler zaten RAM'de ise onları kullanır; yoksa CSV'den okur.
def safe_get(name, path):
    return globals()[name] if name in globals() else pd.read_csv(path)

WDI_CSV     = safe_get("WDI_CSV", "WDICSV.csv")
WDI_country = safe_get("WDI_country", "WDICountry.csv")

year_cols = [c for c in WDI_CSV.columns if str(c).isdigit()]
WDI_long = WDI_CSV.melt(
    id_vars=["Country Name","Country Code","Indicator Name","Indicator Code"],
    value_vars=year_cols, var_name="Year", value_name="Value"
)
WDI_long["Year"]  = pd.to_numeric(WDI_long["Year"], errors="coerce").astype("Int64")
WDI_long["Value"] = pd.to_numeric(WDI_long["Value"], errors="coerce")

# pandemi etkisini dışarıda bırakalım: 2000–2020
WDI_filtered = WDI_long[(WDI_long["Year"]>=2000) & (WDI_long["Year"]<=2020)].copy()


In [10]:
TARGET = "GDP growth (annual %)"

CANDIDATES = [
    # Demografi / nüfus dinamiği
    "Population growth (annual %)",
    "Fertility rate, total (births per woman)",
    "Net migration",
    # Sağlık
    "Life expectancy at birth, total (years)",
    "Death rate, crude (per 1,000 people)",
    "Health expenditure, total (% of GDP)",
    # Eğitim / insan sermayesi
    "School enrollment, secondary (% gross)",
    "School enrollment, tertiary (% gross)",
    # Altyapı / dijital
    "Access to electricity (% of population)",
    "Individuals using the Internet (% of population)",
    # Ticaret / makro
    "Trade (% of GDP)",
    "Unemployment, total (% of total labor force) (national estimate)",
    "Inflation, consumer prices (annual %)",
    # Çevre / kaynak
    "CO2 emissions (metric tons per capita)",
    "Renewable energy consumption (% of total final energy consumption)",
    "Forest area (% of land area)"
]

present = set(WDI_filtered["Indicator Name"].dropna().unique())
USE = [c for c in CANDIDATES if c in present]
print(f"Kullanılabilir özellik sayısı: {len(USE)} / {len(CANDIDATES)}")
print("Örnek özellikler:", USE[:8])


Kullanılabilir özellik sayısı: 14 / 16
Örnek özellikler: ['Population growth (annual %)', 'Fertility rate, total (births per woman)', 'Net migration', 'Life expectancy at birth, total (years)', 'Death rate, crude (per 1,000 people)', 'School enrollment, secondary (% gross)', 'School enrollment, tertiary (% gross)', 'Access to electricity (% of population)']


In [13]:
# --- Bölge / gelir eşlemesi için sütun adı düzeltmesi ---
# Bazı WDI versiyonlarında sütun "TableName" yerine "Country Name" olabilir.
if "TableName" not in WDI_country.columns:
    possible = [c for c in WDI_country.columns if "Name" in c]
    print("⚠️  'TableName' bulunamadı, bunun yerine şu sütun kullanılacak:", possible[0])
    WDI_country = WDI_country.rename(columns={possible[0]: "TableName"})

# Bölge ve gelir grubu ekle
WDI_country["TableName"] = WDI_country["TableName"].astype(str)
dfi = dfi.merge(
    WDI_country[["TableName","Region","IncomeGroup"]],
    left_on="Country Name", right_on="TableName", how="left"
).drop(columns=["TableName"])



NEEDED = [TARGET] + USE
sub = WDI_filtered[WDI_filtered["Indicator Name"].isin(NEEDED)].copy()

pivot = sub.pivot_table(values="Value", index=["Country Name","Year"], columns="Indicator Name").reset_index()

# tip düzeltme
pivot["Year"] = pd.to_numeric(pivot["Year"], errors="coerce").astype("Int64")
num_cols = [c for c in pivot.columns if c not in ["Country Name","Year"]]
pivot[num_cols] = pivot[num_cols].apply(pd.to_numeric, errors="coerce")

# ülke içinde yıl sırasına göre lineer interpolate (yalnızca sayısallar)
def country_interp(g):
    g = g.sort_values("Year").copy()
    cols = [c for c in g.columns if c not in ["Country Name","Year"]]
    g[cols] = g[cols].interpolate(method="linear", limit_direction="both")
    return g

dfi = pivot.groupby("Country Name", group_keys=False).apply(country_interp)

# kalan boşlukları sütun medyanı ile doldur
dfi[num_cols] = dfi[num_cols].fillna(dfi[num_cols].median(numeric_only=True))

# bölge / gelir grubu ekleyelim (hikâye ve hata analizi için)
WDI_country["TableName"] = WDI_country["TableName"].astype(str)
dfi = dfi.merge(
    WDI_country[["TableName","Region","IncomeGroup"]],
    left_on="Country Name", right_on="TableName", how="left"
).drop(columns=["TableName"])

# hedef satırı boş olmasın
dfi = dfi.dropna(subset=[TARGET]).copy()

print(dfi.shape)
dfi.head()


⚠️  'TableName' bulunamadı, bunun yerine şu sütun kullanılacak: Short Name


KeyError: "['IncomeGroup'] not in index"

In [ ]:
FEATURES = [f for f in USE if f in dfi.columns]   # sayısallar
X = dfi[FEATURES].copy()
y = dfi[TARGET].copy()
groups = dfi["Country Name"].astype(str).values   # ülke grupları

# Boru hattı: median imputasyon + standardizasyon + RidgeCV
pre = ColumnTransformer(
    transformers=[("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                    ("sc", StandardScaler())]),
                   FEATURES)],
    remainder="drop"
)

alphas = np.logspace(-3, 3, 13)
model = RidgeCV(alphas=alphas)

pipe = Pipeline([("prep", pre), ("model", model)])

cv = GroupKFold(n_splits=5)
r2_scores  = cross_val_score(pipe, X, y, cv=cv, groups=groups, scoring="r2")
mae_scores = -cross_val_score(pipe, X, y, cv=cv, groups=groups,
                              scoring=make_scorer(mean_absolute_error, greater_is_better=False))

print(f"GroupKFold CV R²: {r2_scores.mean():.3f} ± {r2_scores.std():.3f}")
print(f"GroupKFold CV MAE: {mae_scores.mean():.3f} ± {mae_scores.std():.3f}")

pipe.fit(X, y)
print("Seçilen alpha:", pipe.named_steps["model"].alpha_)


In [ ]:
y_pred = pipe.predict(X)

plt.figure(figsize=(6,6))
plt.scatter(y, y_pred, alpha=0.35)
lims = [min(y.min(), y_pred.min()), max(y.max(), y_pred.max())]
plt.plot(lims, lims, "r--")
plt.xlabel("Gerçek GDP growth (annual %)")
plt.ylabel("Tahmin")
plt.title("Gerçek vs Tahmin (eğitim verisi üzerinde)")
plt.grid(True); plt.show()


In [ ]:
perm = permutation_importance(pipe, X, y, scoring="r2", n_repeats=10, random_state=42)
imp = pd.DataFrame({"feature": FEATURES, "importance": perm.importances_mean}) \
        .sort_values("importance", ascending=False)

sns.barplot(data=imp.head(12), y="feature", x="importance")
plt.title("Özellik Önemleri (permütasyon, R² düşüşü)")
plt.xlabel("Önem (ortalama R² azalması)"); plt.ylabel(""); plt.show()

# bölge bazlı ortalama artık (hikâyeleştirme için)
dfi["pred_growth"] = pipe.predict(X)
dfi["resid"] = dfi[TARGET] - dfi["pred_growth"]

reg_err = dfi.groupby("Region", dropna=False)["resid"].mean().sort_values()
display(reg_err)

sns.barplot(x=reg_err.values, y=reg_err.index)
plt.title("Bölgelere Göre Ortalama Hata (GDP growth, yüzde puan)"); plt.xlabel("Ortalama Artık"); plt.ylabel(""); plt.show()


In [ ]:
top_feats = imp.head(5)["feature"].tolist()
msg = f"""
• 2000–2020 döneminde ülke-bazlı 5-fold CV ile modelimizin performansı: R²≈{r2_scores.mean():.2f}, MAE≈{mae_scores.mean():.2f} yüzde puan.
• Büyüme üzerinde en etkili faktörler: {', '.join(top_feats)} (permütasyon önem sırasına göre).
• Eğitim ve ticaret açıklığı gibi yapısal göstergeler büyüme ile pozitif ilişki gösterirken,
  demografik baskılar (yüksek doğurganlık vb.) ve makro dengesizliklerin eşlik ettiği dönemlerde büyüme sınırlanıyor.
• Enflasyon ve işsizlik ile büyüme arasında ters yönlü ilişki gözlense de,
  bu değişkenleri "büyüme düşüşünün yansıması" olarak yorumluyoruz; doğrudan nedensellik iddiasında bulunmuyoruz.
• Bölgesel hata analizi, modelin bazı bölgelerde sistematik sapma yaptığını gösteriyor
  (grafikteki artı/eksi sapmalar). Bu farklılıklar politika öncelikleri için ipucu veriyor.
"""
print(msg)
